In [1]:
"""
Task 1: Data Cleaning & Preprocessing
AI & ML Internship - Elevate Labs

Dataset : Titanic Dataset
Steps followed (per the task mini-guide):
  1. Import the dataset and explore basic info (nulls, data types)
  2. Handle missing values using mean/median/imputation
  3. Convert categorical features into numerical using encoding
  4. Normalize/standardize the numerical features
  5. Visualize outliers using boxplots and remove them
"""

'\nTask 1: Data Cleaning & Preprocessing\nAI & ML Internship - Elevate Labs\n\nDataset : Titanic Dataset\nSteps followed (per the task mini-guide):\n  1. Import the dataset and explore basic info (nulls, data types)\n  2. Handle missing values using mean/median/imputation\n  3. Convert categorical features into numerical using encoding\n  4. Normalize/standardize the numerical features\n  5. Visualize outliers using boxplots and remove them\n'

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler

In [3]:
sns.set_style("whitegrid")

In [4]:
# ---------------------------------------------------------------------
# STEP 1: Import the dataset and explore basic info (nulls, data types)
# ---------------------------------------------------------------------
print("=" * 70)
print("STEP 1: IMPORT & EXPLORE THE DATASET")
print("=" * 70)

STEP 1: IMPORT & EXPLORE THE DATASET


In [5]:
df = sns.load_dataset("titanic")
df.to_csv("data/titanic_raw.csv", index=False)

In [6]:
print(f"\nShape of dataset: {df.shape}")
print("\nFirst 5 rows:")
print(df.head())


Shape of dataset: (891, 15)

First 5 rows:
   survived  pclass     sex   age  sibsp  parch     fare embarked  class  \
0         0       3    male  22.0      1      0   7.2500        S  Third   
1         1       1  female  38.0      1      0  71.2833        C  First   
2         1       3  female  26.0      0      0   7.9250        S  Third   
3         1       1  female  35.0      1      0  53.1000        S  First   
4         0       3    male  35.0      0      0   8.0500        S  Third   

     who  adult_male deck  embark_town alive  alone  
0    man        True  NaN  Southampton    no  False  
1  woman       False    C    Cherbourg   yes  False  
2  woman       False  NaN  Southampton   yes   True  
3  woman       False    C  Southampton   yes  False  
4    man        True  NaN  Southampton    no   True  


In [7]:
print("\nData types:")
print(df.dtypes)


Data types:
survived          int64
pclass            int64
sex                 str
age             float64
sibsp             int64
parch             int64
fare            float64
embarked            str
class          category
who                 str
adult_male         bool
deck           category
embark_town         str
alive               str
alone              bool
dtype: object


In [8]:
print("\nBasic statistical summary:")
print(df.describe(include="all").T)


Basic statistical summary:
             count unique          top freq       mean        std   min  \
survived     891.0    NaN          NaN  NaN   0.383838   0.486592   0.0   
pclass       891.0    NaN          NaN  NaN   2.308642   0.836071   1.0   
sex            891      2         male  577        NaN        NaN   NaN   
age          714.0    NaN          NaN  NaN  29.699118  14.526497  0.42   
sibsp        891.0    NaN          NaN  NaN   0.523008   1.102743   0.0   
parch        891.0    NaN          NaN  NaN   0.381594   0.806057   0.0   
fare         891.0    NaN          NaN  NaN  32.204208  49.693429   0.0   
embarked       889      3            S  644        NaN        NaN   NaN   
class          891      3        Third  491        NaN        NaN   NaN   
who            891      3          man  537        NaN        NaN   NaN   
adult_male     891      2         True  537        NaN        NaN   NaN   
deck           203      7            C   59        NaN        NaN   NaN 

In [9]:
print("\nMissing values per column:")
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_summary = pd.DataFrame({"missing_count": missing, "missing_%": missing_pct})
print(missing_summary[missing_summary["missing_count"] > 0])


Missing values per column:
             missing_count  missing_%
age                    177      19.87
embarked                 2       0.22
deck                   688      77.22
embark_town              2       0.22


In [10]:
# ---------------------------------------------------------------------
# STEP 2: Handle missing values using mean/median/imputation
# ---------------------------------------------------------------------
print("\n" + "=" * 70)
print("STEP 2: HANDLE MISSING VALUES")
print("=" * 70)


STEP 2: HANDLE MISSING VALUES


In [11]:
# Drop columns that are almost entirely empty or redundant/duplicate info
# 'deck' is ~77% missing -> drop
# 'embark_town' duplicates 'embarked'; 'alive' duplicates 'survived';
# 'class' duplicates 'pclass'; 'who'/'adult_male' duplicate age/sex info.
# We drop the redundant duplicates but keep the core numeric/categorical
# features so encoding & scaling has meaningful columns to work on.
cols_to_drop = ["deck", "embark_town", "alive", "class", "who", "adult_male", "alone"]
df_clean = df.drop(columns=cols_to_drop)
print(f"\nDropped high-missing / redundant columns: {cols_to_drop}")


Dropped high-missing / redundant columns: ['deck', 'embark_town', 'alive', 'class', 'who', 'adult_male', 'alone']


In [12]:
# Age (numeric, some skew) -> impute with MEDIAN
median_age = df_clean["age"].median()
df_clean["age"] = df_clean["age"].fillna(median_age)
print(f"Filled 'age' missing values with median = {median_age}")

Filled 'age' missing values with median = 28.0


In [13]:
# Embarked (categorical, only 2 missing) -> impute with MODE
mode_embarked = df_clean["embarked"].mode()[0]
df_clean["embarked"] = df_clean["embarked"].fillna(mode_embarked)
print(f"Filled 'embarked' missing values with mode = '{mode_embarked}'")

Filled 'embarked' missing values with mode = 'S'


In [14]:
print("\nMissing values after imputation:")
print(df_clean.isnull().sum())


Missing values after imputation:
survived    0
pclass      0
sex         0
age         0
sibsp       0
parch       0
fare        0
embarked    0
dtype: int64


In [15]:
# ---------------------------------------------------------------------
# STEP 3: Convert categorical features into numerical using encoding
# ---------------------------------------------------------------------
print("\n" + "=" * 70)
print("STEP 3: ENCODE CATEGORICAL FEATURES")
print("=" * 70)


STEP 3: ENCODE CATEGORICAL FEATURES


In [16]:
# 'sex' -> binary -> Label Encoding (male/female -> 0/1)
le = LabelEncoder()
df_clean["sex"] = le.fit_transform(df_clean["sex"])
print(f"\nLabel encoded 'sex': {dict(zip(le.classes_, le.transform(le.classes_)))}")


Label encoded 'sex': {'female': np.int64(0), 'male': np.int64(1)}


In [17]:
# 'embarked' -> more than 2 categories, no ordinal relationship -> One-Hot Encoding
df_clean = pd.get_dummies(df_clean, columns=["embarked"], prefix="embarked", drop_first=True)
new_ohe_cols = [c for c in df_clean.columns if c.startswith("embarked_")]
df_clean[new_ohe_cols] = df_clean[new_ohe_cols].astype(int)
print(f"One-hot encoded 'embarked' into: {new_ohe_cols}")

One-hot encoded 'embarked' into: ['embarked_Q', 'embarked_S']


In [18]:
print("\nColumns after encoding:")
print(df_clean.dtypes)


Columns after encoding:
survived        int64
pclass          int64
sex             int64
age           float64
sibsp           int64
parch           int64
fare          float64
embarked_Q      int64
embarked_S      int64
dtype: object


In [19]:
# ---------------------------------------------------------------------
# STEP 4: Normalize / standardize the numerical features
# ---------------------------------------------------------------------
print("\n" + "=" * 70)
print("STEP 4: FEATURE SCALING (STANDARDIZATION)")
print("=" * 70)


STEP 4: FEATURE SCALING (STANDARDIZATION)


In [20]:
numeric_cols = ["age", "fare", "sibsp", "parch"]
scaler = StandardScaler()
df_clean[numeric_cols] = scaler.fit_transform(df_clean[numeric_cols])
print(f"\nStandardized columns (mean=0, std=1): {numeric_cols}")
print(df_clean[numeric_cols].describe().T[["mean", "std"]])


Standardized columns (mean=0, std=1): ['age', 'fare', 'sibsp', 'parch']
               mean       std
age    2.272780e-16  1.000562
fare   3.987333e-18  1.000562
sibsp  4.386066e-17  1.000562
parch  5.382900e-17  1.000562


In [21]:
# ---------------------------------------------------------------------
# STEP 5: Visualize outliers using boxplots and remove them
# ---------------------------------------------------------------------
print("\n" + "=" * 70)
print("STEP 5: OUTLIER DETECTION & REMOVAL")
print("=" * 70)


STEP 5: OUTLIER DETECTION & REMOVAL


In [22]:
# We inspect all four numeric columns visually, but only apply IQR-based
# removal to 'age' and 'fare'. 'sibsp' and 'parch' are discrete COUNT
# variables (mostly 0/1) rather than continuous measurements, so the IQR
# method flags almost every non-zero value as an "outlier" and would strip
# away legitimate data. IQR outlier removal is best suited to continuous
# features, so age & fare are the right targets here.
visualize_cols = ["age", "fare", "sibsp", "parch"]
outlier_removal_cols = ["age", "fare"]

In [23]:
# --- BEFORE removal: boxplots ---
fig, axes = plt.subplots(1, len(visualize_cols), figsize=(16, 4))
for ax, col in zip(axes, visualize_cols):
    sns.boxplot(y=df_clean[col], ax=ax, color="skyblue")
    ax.set_title(f"{col} (before)")
plt.tight_layout()
plt.savefig("images/boxplots_before_outlier_removal.png", dpi=150)
plt.close()
print("\nSaved: images/boxplots_before_outlier_removal.png")


Saved: images/boxplots_before_outlier_removal.png


In [24]:
# IQR method to detect & remove outliers
def remove_outliers_iqr(data, columns):
    data = data.copy()
    for col in columns:
        Q1 = data[col].quantile(0.25)
        Q3 = data[col].quantile(0.75)
        IQR = Q3 - Q1
        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR
        before = len(data)
        data = data[(data[col] >= lower) & (data[col] <= upper)]
        removed = before - len(data)
        print(f"  '{col}': removed {removed} outlier rows (bounds: [{lower:.2f}, {upper:.2f}])")
    return data

In [25]:
print("\nRemoving outliers using IQR method (age & fare only):")
df_final = remove_outliers_iqr(df_clean, outlier_removal_cols)
print(f"\nRows before outlier removal: {len(df_clean)}")
print(f"Rows after outlier removal:  {len(df_final)}")


Removing outliers using IQR method (age & fare only):
  'age': removed 66 outlier rows (bounds: [-2.06, 1.93])
  'fare': removed 107 outlier rows (bounds: [-1.16, 0.63])

Rows before outlier removal: 891
Rows after outlier removal:  718


In [26]:
# --- AFTER removal: boxplots (all 4 columns, to show the effect) ---
fig, axes = plt.subplots(1, len(visualize_cols), figsize=(16, 4))
for ax, col in zip(axes, visualize_cols):
    sns.boxplot(y=df_final[col], ax=ax, color="lightgreen")
    ax.set_title(f"{col} (after)")
plt.tight_layout()
plt.savefig("images/boxplots_after_outlier_removal.png", dpi=150)
plt.close()
print("Saved: images/boxplots_after_outlier_removal.png")

Saved: images/boxplots_after_outlier_removal.png


In [27]:
# ---------------------------------------------------------------------
# Save final cleaned dataset
# ---------------------------------------------------------------------
df_final.to_csv("data/titanic_cleaned.csv", index=False)
print("\n" + "=" * 70)
print(f"Final cleaned dataset saved to data/titanic_cleaned.csv  (shape: {df_final.shape})")
print("=" * 70)


Final cleaned dataset saved to data/titanic_cleaned.csv  (shape: (718, 9))
